<a href="https://colab.research.google.com/github/oluwoleadetifa/movie_genre_classification/blob/dev/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git

%cd movie_genre_classification

import sys
sys.path.append('/content/movie_genre_classification')

fatal: destination path 'movie_genre_classification' already exists and is not an empty directory.
/content/movie_genre_classification


In [2]:
from google.colab import files
uploaded = files.upload()

Saving movies_with_posters.csv to movies_with_posters (3).csv


In [3]:
import os
print(os.listdir("/content"))

['.config', 'movie_genre_classification', 'sample_data']


In [5]:
import os
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir(os.getcwd()))

Current working directory: /content/movie_genre_classification
Files here: ['movies_with_posters (3).csv', 'val (1).csv', 'train.csv', 'helper.txt', 'requirements-lock.txt', 'val.csv', '.git', 'movies_with_posters (1).csv', 'experiments', '.env.example', 'setup.sh', 'docs', 'data', 'README.md', 'movies_with_posters (2).csv', 'movies_with_posters.csv', 'requirements.txt', 'notebooks', 'train (1).csv', 'environment.yml', 'outputs', '.gitignore', 'test.csv', 'test (1).csv', 'src']


In [6]:
import os
import shutil

os.makedirs("/content/movie_genre_classification/data/processed", exist_ok=True)

cwd = os.getcwd()
files_here = os.listdir(cwd)
matches = [f for f in files_here if "movies_with_posters" in f]

print("Matches found:", matches)

file_name = matches[0]

shutil.move(
os.path.join(cwd, file_name),
"/content/movie_genre_classification/data/processed/movies_with_posters.csv"
)

print("Dataset moved correctly:", file_name)

Matches found: ['movies_with_posters (3).csv', 'movies_with_posters (1).csv', 'movies_with_posters (2).csv', 'movies_with_posters.csv']
Dataset moved correctly: movies_with_posters (3).csv


In [7]:
print(os.listdir("/content/movie_genre_classification/data/processed"))

['.gitkeep', 'movies_with_posters.csv']


In [10]:
import os
import shutil

os.makedirs("/content/movie_genre_classification/data/splits", exist_ok=True)

cwd = os.getcwd()
files_here = os.listdir(cwd)

for name in ["train.csv", "val.csv", "test.csv"]:
  matches = [f for f in files_here if f == name or f.startswith(name.replace(".csv", ""))]
print(name, "->", matches)
if matches:
  shutil.move(
os.path.join(cwd, matches[0]),
f"/content/movie_genre_classification/data/splits/{name}"
)

print("Split files moved.")
print(os.listdir("/content/movie_genre_classification/data/splits"))

test.csv -> ['test.csv', 'test (1).csv']
Split files moved.
['train.csv', 'val.csv', 'test.csv']


In [12]:
config_text = """
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"

for folder in [
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    SPLITS_DIR,
    OUTPUT_DIR,
    MODELS_DIR,
    METRICS_DIR,
    FIGURES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_CSV = PROCESSED_DATA_DIR / "movies_with_posters.csv"

TRAIN_SPLIT_CSV = SPLITS_DIR / "train.csv"
VAL_SPLIT_CSV = SPLITS_DIR / "val.csv"
TEST_SPLIT_CSV = SPLITS_DIR / "test.csv"

TEXT_COLUMN = "description"
ID_COLUMN = "movie_id"
LABEL_COLUMN = "label"
GENRE_COLUMN = "genre"
IMAGE_PATH_COLUMN = "image_path"

CLASS_NAMES = ["action", "comedy", "horror", "romance"]
"""

with open("/content/movie_genre_classification/src/config.py", "w") as f:
    f.write(config_text)

print("config.py updated")

config.py updated


In [13]:
import importlib
import src.config
import src.data.load_data
import src.data.make_splits

importlib.reload(src.config)
importlib.reload(src.data.load_data)
importlib.reload(src.data.make_splits)

print("Modules reloaded")

Modules reloaded


In [14]:
from src.data.load_data import load_dataset, basic_dataset_check

df = load_dataset()
basic_dataset_check(df)
df.head()

Dataset shape: (687, 6)

Columns:
['movie_id', 'description', 'genre', 'image_path', 'image_exists', 'label']

Missing descriptions:
0

Genre distribution:
label
2    250
0    239
1    106
3     92
Name: count, dtype: int64


,movie_id,description,genre,image_path,image_exists,label
0,tt1798632,A young girl tries to understand how she myste...,horror,/Users/oluwoleadetifa/Library/CloudStorage/Goo...,True,2
1,tt9214832,"In 1800s England, a well meaning but selfish y...",comedy,/Users/oluwoleadetifa/Library/CloudStorage/Goo...,True,1
2,tt21249656,Olga and Maks are 15 years apart. She is a suc...,romance,/Users/oluwoleadetifa/Library/CloudStorage/Goo...,True,3
3,tt7149730,A reformed hunter living in isolation on a wil...,action,/Users/oluwoleadetifa/Library/CloudStorage/Goo...,True,0
4,tt3876910,A reformed sociopath journeys to a remote isla...,action,/Users/oluwoleadetifa/Library/CloudStorage/Goo...,True,0


In [15]:
print(df.columns.tolist())
print(df[["movie_id", "description", "genre", "label"]].head())

['movie_id', 'description', 'genre', 'image_path', 'image_exists', 'label']
     movie_id                                        description    genre  \
0   tt1798632  A young girl tries to understand how she myste...   horror   
1   tt9214832  In 1800s England, a well meaning but selfish y...   comedy   
2  tt21249656  Olga and Maks are 15 years apart. She is a suc...  romance   
3   tt7149730  A reformed hunter living in isolation on a wil...   action   
4   tt3876910  A reformed sociopath journeys to a remote isla...   action   

   label  
0      2  
1      1  
2      3  
3      0  
4      0  
